# 04. Creating Custom Modules

## Learning objectives
- Create reusable Python modules in `.py` files.
- Import custom functions, variables, and classes.
- Choose between module imports and selected-member imports.
- Organize a small application into separate files.
- Explain `__name__`, the main guard, module initialization, and caching.
- Recognize common import errors and basic circular-import risks.

This notebook applies the import techniques from `02_Import_Statements.ipynb` to code you create yourself. The next notebook explores the main-guard pattern in greater depth.

## 1. Creating Your First Custom Module

A custom module is a Python file you create. Its import name is usually the filename without `.py`.

Create `calculator.py`:

```python
PI = 3.14159

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b
```

This file contains a constant and four reusable functions. Save it beside the file that will import it.

### Demonstration in this notebook

The following cell defines equivalent functions so they can be run here. It does not create an external `calculator.py` file.

In [ ]:
PI = 3.14159

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

print(add(10, 5))
print(subtract(10, 5))
print(multiply(10, 5))
print(divide(10, 5))
print(PI)

## 2. Importing Your Custom Module

Create `main.py` beside `calculator.py`:

```python
import calculator

print(calculator.add(10, 5))
print(calculator.subtract(10, 5))
print(calculator.multiply(10, 5))
print(calculator.divide(10, 5))
print(calculator.PI)
```

Expected output:

```text
15
5
50
2.0
3.14159
```

The prefix `calculator.` makes the source of each name clear. You do not need to copy function definitions into `main.py`.

### Import selected members

```python
from calculator import add, divide

print(add(20, 30))
print(divide(20, 4))
```

Expected output: `50` and `5.0`.

An alias is also possible:

```python
import calculator as calc
print(calc.multiply(6, 7))
```

Expected output: `42`.

These styles were introduced in `02_Import_Statements.ipynb`; here, the emphasis is using them with your own modules.

## 3. Importing Variables, Functions, and Classes

Create `student.py`:

```python
COLLEGE = "ABC College"

def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):
        return calculate_average(self.marks)
```

Then in `main.py`:

```python
from student import COLLEGE, Student, calculate_average

print(COLLEGE)
print(calculate_average([80, 90, 100]))

student = Student("Asha", [80, 90, 100])
print(student.name)
print(student.average())
```

This imports a module-level constant, a function, and a class, then creates an object from the imported class.

## 4. Building a Multi-File Project

A student marks application can separate calculation logic from application code:

```text
student_project/
├── marks_utils.py
└── main.py
```

`marks_utils.py`:

```python
def calculate_total(marks):
    return sum(marks)

def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

def find_highest(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return max(marks)
```

`main.py`:

```python
from marks_utils import (
    calculate_total,
    calculate_average,
    find_highest,
)

marks = [85, 90, 78, 92, 88]
print("Total:", calculate_total(marks))
print("Average:", calculate_average(marks))
print("Highest:", find_highest(marks))
```

Expected output:

```text
Total: 433
Average: 86.6
Highest: 92
```

Separating calculation logic makes it reusable, easier to test, and easier to maintain.

### Run the marks example in this notebook

In [ ]:
def calculate_total(marks):
    return sum(marks)

def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

def find_highest(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return max(marks)

marks = [85, 90, 78, 92, 88]
print("Total:", calculate_total(marks))
print("Average:", calculate_average(marks))
print("Highest:", find_highest(marks))

## 5. The `__name__` Variable and Main Guard

Python assigns a special variable named `__name__` to each module.

- When a file is run directly as the main program, `__name__` is `"__main__"`.
- When a file is imported, `__name__` is normally the module's import name.

Example `greetings.py`:

```python
def greet(name):
    return f"Hello, {name}!"

if __name__ == "__main__":
    print(greet("Python Learner"))
```

The greeting runs when `greetings.py` is executed directly, but not simply because another file imports it. The function remains available to importing code. The next notebook explores this pattern in greater depth.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

print(greet("Python Learner"))
print("__name__ in this notebook cell:", __name__)

## 6. Module Execution and Reloading

On the first import in a process, Python executes a module's top-level statements to initialize it. Python normally caches the module in `sys.modules`, so later imports reuse the loaded module rather than executing it again.

`importlib.reload()` reloads a module explicitly, but existing references to objects from that module may not automatically be rebound. Reloading is not a general substitute for restarting a process or notebook kernel.

In [ ]:
import math
import sys
import importlib

print("math module name:", math.__name__)
print("math cached in sys.modules:", "math" in sys.modules)

reloaded_math = importlib.reload(math)
print("Reload returned the same module object:", reloaded_math is math)

## 7. Common Mistakes and Debugging

| Mistake | How to avoid it |
|---|---|
| Wrong filename or import spelling | Check the `.py` filename and import name carefully. |
| Running from an unexpected directory | Check the working directory and module search path. |
| Naming a file `math.py` or `random.py` | Choose a distinctive filename to avoid shadowing other modules. |
| Putting startup code at module level | Use a main guard when code should run only on direct execution. |
| Duplicating functions across files | Put reusable logic in a shared module. |
| Creating circular imports | Reduce dependencies or move shared logic into a separate module. |

`ModuleNotFoundError` means Python cannot locate the module or a dependency. `ImportError` means an import could not be completed. Circular imports can expose partially initialized modules and cause missing-name errors.

In [ ]:
import sys
from pathlib import Path

print("Working directory:", Path.cwd())
print("First search path entry:", sys.path[0] if sys.path else "(empty)")

## 8. Practical Project: Calculator and Student Marks Utilities

Create these files in one folder:

```text
python_utilities/
├── calculator.py
├── marks_utils.py
└── main.py
```

`calculator.py`:

```python
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b
```

`marks_utils.py`:

```python
def calculate_total(marks):
    return sum(marks)

def calculate_average(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return sum(marks) / len(marks)

def find_highest(marks):
    if not marks:
        raise ValueError("Marks cannot be empty")
    return max(marks)
```

`main.py`:

```python
from calculator import add, divide, multiply, subtract
from marks_utils import calculate_average, calculate_total, find_highest

def main():
    print("Addition:", add(10, 5))
    print("Division:", divide(20, 4))

    marks = [85, 90, 78, 92, 88]
    print("Total:", calculate_total(marks))
    print("Average:", calculate_average(marks))
    print("Highest:", find_highest(marks))

if __name__ == "__main__":
    main()
```

Run from that directory with `python main.py`. The main guard makes the application run when `main.py` is executed directly.

## Key Takeaways

- A custom module is a `.py` file whose definitions can be imported and reused.
- Its import name is usually the filename without `.py`.
- You can import a module, selected members, or aliases.
- Separating reusable logic from application flow reduces duplication.
- `__name__ == "__main__"` distinguishes direct execution from ordinary import.
- Top-level code executes during initial import; modules are generally cached in `sys.modules`.
- Check filenames, working directories, search paths, and dependencies when debugging imports.